# OAG_FABLE — Agentic Intelligence on Snowflake · Interactive Companion

The executable twin of [`docs/snowflake-agentic-primer.html`](../docs/snowflake-agentic-primer.html).
It drives the **live `OAG_FABLE` database** with the Snowflake Python driver — top-down, the way a
request falls through the agent stack:

1. **The objects as built** — schemas, canonical tables, row counts.
2. **The semantic model** — Snowflake Semantic Views; reproduce the hero gold through `SEMANTIC_VIEW()`.
3. **The knowledge graph** — `KG_NODE`/`KG_EDGE` and the recursive-CTE traversal tools.
4. **The governed tools** — the owner's-rights UDF/UDTF surface the agent may call.
5. **The agent** — inspect `PRODUCTION_AGENT`, then invoke the client-side `ProductionAgent` and the
   delegated Cortex Agent over REST (`:run`); grade both against the frozen gold.
6. **The governance boundary** — the three RBAC roles.

> Read-only except for warehouse compute + a couple of Cortex calls. It uses the `snow` CLI connection
> **`proton`** (key-pair auth) exactly as `snowflake/agent/agent.py` does — no credentials in the repo.

## 0 · Setup — connect with the Snowflake driver

The connection parameters (account, user, warehouse, role, key-pair) are read from the `snow` CLI's
`config.toml`, the same way the agent does. `q(sql)` returns a `pandas.DataFrame` for display.

In [1]:
import os, sys, json, tomllib, subprocess, urllib.request
from pathlib import Path
import pandas as pd
import snowflake.connector

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
GOLD = REPO / 'data' / 'frozen' / 'gold'
CONNECTION = os.environ.get('OAG_SNOWFLAKE_CONNECTION', 'proton')
CONFIG = os.path.expanduser('~/Library/Application Support/snowflake/config.toml')

def _params(name=CONNECTION):
    with open(CONFIG, 'rb') as f:
        p = dict(tomllib.load(f)['connections'][name])
    p.pop('database', None); p.pop('schema', None)
    return p

con = snowflake.connector.connect(**_params())

def q(sql, params=()):
    cur = con.cursor()
    try:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    finally:
        cur.close()

q("SELECT CURRENT_ACCOUNT() AS account, CURRENT_ROLE() AS role, "
  "CURRENT_WAREHOUSE() AS warehouse, CURRENT_VERSION() AS snowflake_version")

,ACCOUNT,ROLE,WAREHOUSE,SNOWFLAKE_VERSION
0,KJ94708,SYSADMIN,COMPUTE_WH,10.23.103


## 1 · The objects as built  <code>OAG_FABLE</code>

One database, three schemas: **CANONICAL** (OSDU/PDM system-of-record), **SEMANTIC** (governed metric
views), **KNOWLEDGE** (the LPG graph + the agent's governed tools + the Cortex Agent).

In [2]:
q("SHOW TERSE SCHEMAS IN DATABASE OAG_FABLE")[['name']]

,name
0,CANONICAL
1,INFORMATION_SCHEMA
2,KNOWLEDGE
3,PUBLIC
4,SEMANTIC


The canonical layer is 9 Parquet-loaded tables, stamped with the frozen `config_hash 12a110eecfe2`.
Row counts are the load sanity-check from `sql/02_load_canonical.sql`.

In [3]:
CANON = ['WELL_VOL_DAILY','PRODUCT_VOLUME_SUMMARY','REPORTING_ENTITY','WELL','FIELD',
         'FACILITY','DOWN_TIME_EVENT','WELL_TEST','RPEN_ALLOCATION_FACTOR']
counts = ' UNION ALL '.join(
    f"SELECT '{t}' AS table_name, COUNT(*) AS n_rows FROM OAG_FABLE.CANONICAL.{t}" for t in CANON)
q(counts + ' ORDER BY n_rows DESC')

,TABLE_NAME,N_ROWS
0,WELL_VOL_DAILY,8208
1,PRODUCT_VOLUME_SUMMARY,8208
2,DOWN_TIME_EVENT,282
3,WELL_TEST,251
4,REPORTING_ENTITY,21
5,WELL,18
6,RPEN_ALLOCATION_FACTOR,18
7,FACILITY,6
8,FIELD,3


A peek at the `WELL` dimension — note the human-readable UWI, the Well→Field→Operator hierarchy keys,
and the facility link the knowledge graph is built from.

In [4]:
q("SELECT WELL_ID, UWI, WELL_NAME, FIELD_ID, FIELD_NAME, FACILITY_ID, OPERATOR "
  "FROM OAG_FABLE.CANONICAL.WELL ORDER BY WELL_ID LIMIT 6")

,WELL_ID,UWI,WELL_NAME,FIELD_ID,FIELD_NAME,FACILITY_ID,OPERATOR
0,1,NO 15/9-F-1,Volve-1,1,Volve,1,Equinor
1,2,NO 15/9-F-2,Volve-2,1,Volve,2,Equinor
2,3,NO 15/9-F-3,Volve-3,1,Volve,1,Equinor
3,4,NO 15/9-F-4,Volve-4,1,Volve,2,Equinor
4,5,NO 15/9-F-5,Volve-5,1,Volve,1,Equinor
5,6,NO 15/9-F-6,Volve-6,1,Volve,2,Equinor


## 2 · The semantic model — Snowflake Semantic Views  <code>sql/03</code>

Six governed metric objects compiled from the neutral OSI YAML (`semantic/metrics.yaml` +
`semantic_models.yaml`). Agents *select* these metrics; they never author aggregation SQL.

In [5]:
q("SHOW SEMANTIC VIEWS IN SCHEMA OAG_FABLE.SEMANTIC")[['name','comment']]

,name,comment
0,ALLOCATION,OSI allocation fact (RPEN_ALLOCATION_FACTOR): ...
1,DOWNTIME,OSI downtime fact (DOWN_TIME_EVENT): governed ...
2,FORECAST_VOLUMES,OSI forecast fact (PRODUCT_VOLUME_SUMMARY): ex...
3,SURVEILLANCE,"OSI production-surveillance signal (theme 1, h..."
4,WELL_TESTS,OSI well-test fact (WELL_TEST): governed well_...
5,WELL_VOLUMES,OSI actuals fact (WELL_VOL_DAILY): governed vo...


`DESC SEMANTIC VIEW` exposes the model shape — logical **tables** (with primary keys), typed
**relationships**, **dimensions**, and **metrics**. Note that the derived metrics reference the base
metrics (`production_efficiency = actual_oil / expected_oil`) — Snowflake rejects `SUM(metric)`, which
makes this a closer match to how OSI defines `ratio`/`derived` types.

In [6]:
desc = q("DESC SEMANTIC VIEW OAG_FABLE.SEMANTIC.SURVEILLANCE")
desc[desc['object_kind'] == 'METRIC'][['object_name','property','property_value']].query("property == 'EXPRESSION'")[['object_name','property_value']].rename(columns={'object_name':'metric','property_value':'expression'})

,metric,expression
10,ACTUAL_OIL,SUM(surv.actual_oil_bbl)
14,EXPECTED_OIL,SUM(surv.expected_oil_bbl)
18,OIL_SHORTFALL,surv.expected_oil - surv.actual_oil
22,PRODUCTION_EFFICIENCY,"surv.actual_oil / NULLIF(surv.expected_oil, 0)"


### Reproduce the hero gold through `SEMANTIC_VIEW()`

You name the dimensions and metrics inside the `SEMANTIC_VIEW()` table function; the **row-level filter
goes inside** its `WHERE`. This is theme 1 — wells below 90% of expected oil over the frozen
surveillance week — and it must reproduce `gold/surveillance.json` exactly.

In [7]:
surv = q('''
SELECT well_id, expected_oil AS expected_oil_bbl, actual_oil AS actual_oil_bbl,
       oil_shortfall AS shortfall_bbl, production_efficiency AS efficiency
FROM SEMANTIC_VIEW(OAG_FABLE.SEMANTIC.SURVEILLANCE
  DIMENSIONS surv.well_id
  METRICS surv.expected_oil, surv.actual_oil, surv.oil_shortfall, surv.production_efficiency
  WHERE surv.volume_date BETWEEN '2025-03-25' AND '2025-03-31')
WHERE production_efficiency < 0.9
ORDER BY oil_shortfall DESC, well_id''')
surv

,WELL_ID,EXPECTED_OIL_BBL,ACTUAL_OIL_BBL,SHORTFALL_BBL,EFFICIENCY
0,12,82985.094,50794.13,32190.964,0.612087
1,11,108594.018,76954.91,31639.108,0.708648
2,10,66517.182,43479.05,23038.132,0.653651
3,2,39240.377,17117.53,22122.847,0.436222
4,7,68066.376,46380.24,21686.136,0.681397
5,13,59986.464,38548.77,21437.694,0.642624
6,6,12421.417,3961.53,8459.887,0.318927
7,1,12994.179,4686.85,8307.329,0.360688
8,17,65977.495,58073.41,7904.085,0.880200
9,8,10512.899,4934.21,5578.689,0.469348


Grade the semantic-view result against the co-generated gold — exact flagged set, values within `1e-6`.

In [8]:
gold = json.loads((GOLD / 'surveillance.json').read_text())['flagged']
gkeys = ['expected_oil_bbl','actual_oil_bbl','shortfall_bbl','efficiency']
g = {int(r['well_id']): r for r in gold}
s = {int(r.WELL_ID): r for r in surv.itertuples()}
def close(a,b): return abs(float(a)-float(b)) <= 1e-6*max(abs(float(a)),abs(float(b)),1.0)
missing = set(g)-set(s); extra = set(s)-set(g)
mism = [(w,k) for w in set(g)&set(s) for k in gkeys
        if not close(getattr(s[w], {'expected_oil_bbl':'EXPECTED_OIL_BBL','actual_oil_bbl':'ACTUAL_OIL_BBL',
                                    'shortfall_bbl':'SHORTFALL_BBL','efficiency':'EFFICIENCY'}[k]), g[w][k])]
print(f'flagged: {len(s)}/{len(g)}   missing={missing or None}  extra={extra or None}  mismatches={mism or None}')
print('PASS — semantic view reproduces gold exactly' if not (missing or extra or mism) else 'FAIL')

flagged: 12/12   missing=None  extra=None  mismatches=None
PASS — semantic view reproduces gold exactly


## 3 · The knowledge graph — an explicit, queryable LPG  <code>sql/08</code>

`KG_NODE`/`KG_EDGE` hold every business concept and relationship **once, account-wide**, derived
deterministically from the canonical tables — so graph and substrate cannot drift.

In [9]:
q('''SELECT node_type, COUNT(*) AS nodes FROM OAG_FABLE.KNOWLEDGE.KG_NODE
     GROUP BY 1 ORDER BY nodes DESC''')

,NODE_TYPE,NODES
0,reporting_entity,21
1,well,18
2,facility,6
3,field,3
4,operator,3


In [10]:
q('''SELECT edge_type, COUNT(*) AS edges FROM OAG_FABLE.KNOWLEDGE.KG_EDGE
     GROUP BY 1 ORDER BY edges DESC''')

,EDGE_TYPE,EDGES
0,IN_FIELD,24
1,OPERATED_BY,24
2,REPORTS_ON,21
3,AT_FACILITY,18


### Entity resolution and traversal — the four graph tools

`KG_RESOLVE` maps an NL mention to a node via name / synonym / UWI. `KG_TRAVERSE` walks everything
within N hops (edges bidirectional, cycles guarded). `KG_PATH` finds one shortest path — and reports
true non-connectivity rather than inventing a route.

In [11]:
q("SELECT * FROM TABLE(OAG_FABLE.KNOWLEDGE.KG_RESOLVE('the volve'))")

,NODE_ID,NODE_TYPE,NAME,MATCHED_ON
0,field:1,field,Volve,synonym


`KG_TRAVERSE('field:1', 2)` — everything belonging to the Volve field within two hops: its two
facilities, its wells, their reporting entities, and its operator, each at its minimum depth.

In [12]:
q("SELECT node_type, name, depth FROM TABLE(OAG_FABLE.KNOWLEDGE.KG_TRAVERSE('field:1', 2)) "
  "ORDER BY depth, node_type, name")

,NODE_TYPE,NAME,DEPTH
0,facility,Volve Battery 1,1
1,facility,Volve Battery 2,1
2,reporting_entity,Volve,1
3,well,Volve-1,1
4,well,Volve-2,1
5,well,Volve-3,1
6,well,Volve-4,1
7,well,Volve-5,1
8,well,Volve-6,1
9,operator,Equinor,2


Two `KG_PATH` calls: a well **is** operated by AkerBP (one hop); the same well to a *different*
operator is **empty** — each field has a single operator, so the graph has three disconnected
components and the pathfinder says so honestly.

In [13]:
connected = q("SELECT hops, path FROM TABLE(OAG_FABLE.KNOWLEDGE.KG_PATH('well:12','operator:AkerBP',6))")
disconnected = q("SELECT hops, path FROM TABLE(OAG_FABLE.KNOWLEDGE.KG_PATH('well:12','operator:Equinor',6))")
print('well:12 -> operator:AkerBP :', connected.to_dict('records') or 'NOT CONNECTED')
print('well:12 -> operator:Equinor:', disconnected.to_dict('records') or 'NOT CONNECTED (empty, as expected)')

well:12 -> operator:AkerBP : [{'HOPS': 1, 'PATH': '[\n  "well:12",\n  "-[OPERATED_BY]->",\n  "operator:AkerBP"\n]'}]
well:12 -> operator:Equinor: NOT CONNECTED (empty, as expected)


## 4 · The governed tool surface  <code>KNOWLEDGE.*</code>

The agent's 'context' is the set of governed functions it is allowed to call — owner's-rights
UDF/UDTFs that each reproduce the deterministic reference-compile. Here the hero theme-1 tool and the
theme-2 deferment tool, invoked directly as `TABLE(fn(args))`.

In [14]:
q("SELECT well_id, uwi, ROUND(shortfall_bbl,1) AS shortfall_bbl, ROUND(efficiency,4) AS efficiency "
  "FROM TABLE(OAG_FABLE.KNOWLEDGE.SURVEILLANCE_BELOW_EXPECTED(0))")   # 0 = whole fleet

,WELL_ID,UWI,SHORTFALL_BBL,EFFICIENCY
0,12,NO 15/9-F-12,32191.0,0.6121
1,11,NO 15/9-F-11,31639.1,0.7086
2,10,NO 15/9-F-10,23038.1,0.6537
3,2,NO 15/9-F-2,22122.8,0.4362
4,7,NO 15/9-F-7,21686.1,0.6814
5,13,NO 15/9-F-13,21437.7,0.6426
6,6,NO 15/9-F-6,8459.9,0.3189
7,1,NO 15/9-F-1,8307.3,0.3607
8,17,NO 15/9-F-17,7904.1,0.8802
9,8,NO 15/9-F-8,5578.7,0.4693


In [15]:
q("SELECT cause, ROUND(deferred_oil_bbl,1) AS deferred_oil_bbl, downtime_hours, n_events "
  "FROM TABLE(OAG_FABLE.KNOWLEDGE.DEFERMENT_BY_CAUSE())")

,CAUSE,DEFERRED_OIL_BBL,DOWNTIME_HOURS,N_EVENTS
0,Facility Constraint,24191.5,96.16,7
1,Weather,15093.8,46.99,4
2,ESP Failure,14823.4,59.40,4
3,Power Outage,11091.0,50.99,3
4,Planned Maintenance,11051.9,47.44,3


## 5 · The agent — inspect it, then invoke it  <code>sql/07 · agent/agent.py</code>

`PRODUCTION_AGENT` is a first-class Cortex **Agent** object: it plans, calls the governed tools, and
narrates — but the graded values are always assembled from the *tool result*, never the LLM prose.

In [16]:
q("SHOW AGENTS IN SCHEMA OAG_FABLE.KNOWLEDGE")[['name','comment']]

,name,comment
0,PRODUCTION_AGENT,Delegated production-analytics agent over the ...


### 5a · Client-side — the Python `ProductionAgent`

Cortex `COMPLETE` routes the NL question to one governed tool; the connector executes it; code
assembles the answer-submission contract. (This makes a real Cortex call, so it takes a few seconds.)

In [17]:
sys.path.insert(0, str(REPO / 'snowflake' / 'agent'))
from agent import ProductionAgent

pa = ProductionAgent()
sub = pa.answer('Which wells are producing below expected oil rate this week, and by how much?')
print('question_id :', sub['question_id'])
print('behavior    :', sub['behavior'])
print('answer      :', sub['answer'])
print('provenance  :', sub['provenance'])
print('flagged rows:', len(sub['key_values'].get('flagged', [])))
pd.DataFrame(sub['key_values']['flagged']).head()

question_id : surveillance-below-expected-oil
behavior    : answered
answer      : This query identifies wells across the fleet that are producing below their expected oil rate during the current surveillance week and quantifies the shortfall.
provenance  : {'metrics': ['production_efficiency', 'expected_oil', 'actual_oil', 'oil_shortfall'], 'dimensions': ['well', 'volume_date'], 'filters': [], 'entities': ['Well']}
flagged rows: 12


,well_id,expected_oil_bbl,actual_oil_bbl,shortfall_bbl,efficiency
0,12,82985.094,50794.13,32190.964,0.612087
1,11,108594.018,76954.91,31639.108,0.708648
2,10,66517.182,43479.05,23038.132,0.653651
3,2,39240.377,17117.53,22122.847,0.436222
4,7,68066.376,46380.24,21686.136,0.681397


Grade the agent's own answer against gold — the deterministic-answer path means it matches the
semantic-view result and the gold exactly.

In [18]:
agold = json.loads((GOLD / 'surveillance.json').read_text())['flagged']
ag = {int(r['well_id']): r for r in agold}
asub = {int(r['well_id']): r for r in sub['key_values']['flagged']}
amis = set(ag)-set(asub); aext = set(asub)-set(ag)
amism = [(w,k) for w in set(ag)&set(asub) for k in gkeys if not close(asub[w][k], ag[w][k])]
print(f'agent flagged: {len(asub)}/{len(ag)}   ' + ('PASS' if not (amis or aext or amism) else f'FAIL {amis} {aext} {amism[:3]}'))

agent flagged: 12/12   PASS


Honest abstention is preserved — an off-topic question resolves to no governed tool and the agent
asks for clarification rather than guessing.

In [19]:
off = pa.answer('What is the weather in Stavanger today?')
print('behavior:', off['behavior'])
print('answer  :', off['answer'])

behavior: clarification-requested
answer  : This question is about weather and is outside the scope of production analytics tools.


### 5b · Delegated — drive the deployed Cortex Agent over REST (`:run`)

The seam an **external orchestrator** calls: POST the question to the agent's `:run` endpoint with a
key-pair JWT. The platform plans and executes the governed tools server-side and returns their rows in
a `tool_result` item; we read the graded values from that payload. (Guarded so a JWT/network hiccup
doesn't break the notebook.)

In [20]:
def _jwt():
    out = subprocess.run(['snow','connection','generate-jwt','-c',CONNECTION],
                         capture_output=True, text=True, check=True).stdout
    return next(t.strip() for t in out.split() if t.startswith('eyJ'))

def run_agent(question):
    host = _params()['account'] + '.snowflakecomputing.com'
    url = f'https://{host}/api/v2/databases/OAG_FABLE/schemas/KNOWLEDGE/agents/PRODUCTION_AGENT:run'
    body = json.dumps({'stream': False, 'messages': [
        {'role':'user','content':[{'type':'text','text':question}]}]}).encode()
    req = urllib.request.Request(url, data=body, method='POST', headers={
        'Authorization': f'Bearer {_jwt()}',
        'X-Snowflake-Authorization-Token-Type': 'KEYPAIR_JWT',
        'Content-Type':'application/json','Accept':'application/json'})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

try:
    resp = run_agent('How is well NO 15/9-F-12 related to AkerBP?')
    texts, tools = [], []
    for item in resp.get('content', []):
        if item['type'] == 'text': texts.append(item['text'].strip())
        elif item['type'] == 'tool_result':
            tools.append(item['tool_result'].get('name'))
    print('tools the agent planned:', tools)
    print('agent answer           :', texts[-1] if texts else '(none)')
except Exception as e:
    print('REST :run skipped ->', type(e).__name__, str(e)[:160])

tools the agent planned: ['kg_resolve', 'kg_resolve', 'kg_path']
agent answer           : Well NO 15/9-F-12 (UWI Sleipner-6) is operated by AkerBP — it connects directly to the operator via a single OPERATED_BY relationship.


## 6 · The governance boundary  <code>sql/06</code>

Three least-privilege roles. The decisive Snowflake property: because SQL UDFs run with the **owner's**
rights, `OAG_AGENT` can call every governed tool yet cannot `SELECT` a single raw table — the governed
tools *are* the security boundary.

In [21]:
try:
    g = q("SHOW GRANTS TO ROLE OAG_AGENT")
    display(g[['privilege','granted_on','name']].head(20))
except Exception as e:
    print('need a role that can inspect OAG_AGENT grants ->', str(e)[:120])

,privilege,granted_on,name
0,USAGE,CORTEX_AGENT,OAG_FABLE.KNOWLEDGE.PRODUCTION_AGENT
1,USAGE,DATABASE,OAG_FABLE
2,USAGE,DATABASE_ROLE,SNOWFLAKE.CORTEX_USER
3,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DECLINE_WELLS_FASTER()
4,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DECLINE_WELLS_FASTER_ROWS()
5,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DEFERMENT_BY_CAUSE()
6,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DEFERMENT_BY_CAUSE_ROWS()
7,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DEFERMENT_SUMMARY()
8,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.DEFERMENT_SUMMARY_ROWS()
9,USAGE,FUNCTION,OAG_FABLE.KNOWLEDGE.KG_NEIGHBORS(VARCHAR)


---
### Where to go next

- **`docs/snowflake-agentic-primer.html`** — the narrative map this notebook executes.
- **`snowflake/README.md`** / **`snowflake/agent/README.md`** — the full port + REST findings.
- **`snowflake/COMPARISON.md`** — Databricks vs Snowflake, feature by feature.
- Run every theme end-to-end over REST: `cd snowflake/agent && python test_agent_rest.py`.

In [22]:
con.close()